# 画像変数化・キャラクター一貫性定義パイプライン（立ち絵・アニメーション向け）

アニメーションや立ち絵差分において、画像の状態（構図・ポーズ・向き・表情・視線・外観）を一貫した数値変数として定義・管理するためのノートブックです。  
`pose-alignment.ipynb` と同じ Colab + `rtmlib` (`onnxruntime-gpu`) 基盤に **WD14-Tagger (`SmilingWolf/wd-eva02-large-tagger-v3`)** を統合し、以下のパイプラインを一貫して実行できます。

1. **セットアップ・データ準備**: `rtmlib` / `onnxruntime-gpu` / `huggingface_hub` 等の導入と Drive からの画像コピー
2. **モデル初期化 & スロット定義**: WholeBody（133点骨格）と WD14-Tagger v3 の初期化、表情・視線・構図などの可変スロット定義
3. **画像の変数化（Pose + WD14）**:
   - **骨格変数（Pose）**: 画角・重心・顔の向き（Yaw/Roll）・肩/体幹の傾き・スケール不変プロポーション（頭身比等）・関節角度・正規化骨格座標
   - **タグ変数（WD14）**: 表情・視線・構図・ポーズ・背景の信頼度スコア（`0.0〜1.0` の連続値）
4. **キャラクター固有分布（Character Profile）の構築**: 複数の LoRA 学習データ／リファレンス画像群から、そのキャラ固有の不変特徴（髪・目・衣装・体型プロポーション）の統計分布（平均・標準偏差・出現率）を自動定義
5. **対象画像の変数化 & 一貫性チェック**: 立ち絵差分やアニメフレームを変数化し、キャラ固有分布からのズレ（特徴欠落・外乱混入・頭身崩れ）を自動検知
6. **出力・可視化・Drive保存**: JSON / CSV 出力、分布・逸脱レポート・複数フレーム変数ヒートマップの可視化

## 1. セットアップ

必要なライブラリ（`rtmlib`, `onnxruntime-gpu`, `opencv-python`, `huggingface_hub`, `pandas`）をインストールし、GPUランタイムの状態を確認します。

In [ ]:
%pip install -q rtmlib onnxruntime-gpu opencv-python huggingface_hub pandas matplotlib

In [ ]:
!nvidia-smi

## 2. Google Drive マウント & データ準備

Google Drive をマウントし、以下の作業用ディレクトリを準備します。

- `./data/reference_set/`: **キャラクター固有分布（Character Profile）を作るための複数画像**（LoRA学習データや厳選したLoRA生成画像群）
- `./data/targets/`: **変数化・一貫性チェックを行いたい対象画像**（立ち絵差分やアニメーション用フレーム等）

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Google Drive 上の画像フォルダを作業ディレクトリへコピー（パスは環境に合わせて変更してください）
!mkdir -p ./data/reference_set ./data/targets

# 例: LoRA学習データ（キャラクター固有分布の作成元）
# !cp '/content/drive/MyDrive/Colab Notebooks/lora-dataset/'*.png ./data/reference_set/

# 例: 変数化・一貫性チェックしたい対象画像（立ち絵差分・生成画像など）
# !cp '/content/drive/MyDrive/Colab Notebooks/sample-img/'*.png ./data/targets/

!echo "=== reference_set ===" && ls -1 ./data/reference_set | head -n 20
!echo "=== targets ===" && ls -1 ./data/targets | head -n 20

## 3. モデル初期化 & 可変スロット定義

- **骨格検出**: `rtmlib.Wholebody`（COCO WholeBody 133点、`mode='performance'`）
- **タグ信頼度推論**: `SmilingWolf/wd-eva02-large-tagger-v3`（`onnxruntime-gpu` で直接推論し、`rtmlib` と依存関係を統一）
- **可変スロット定義（`VARIABLE_TAG_SLOTS`）**: アニメーションや立ち絵差分で「意図的に変化させる変数」として扱うタグ群をカテゴリ別に定義します。ここに含まれるタグは、後続の「キャラ固有の不変特徴」自動抽出からは自動的に除外されます。

In [ ]:
import os
import glob
import json
import math
import shutil
from datetime import datetime, timezone

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import onnxruntime as ort
from huggingface_hub import hf_hub_download
from rtmlib import Wholebody, draw_skeleton

# 利用可能な実行プロバイダを自動判定（GPUランタイムなら CUDA、CPUなら CPU にフォールバック）
AVAILABLE_PROVIDERS = ort.get_available_providers()
USE_CUDA = 'CUDAExecutionProvider' in AVAILABLE_PROVIDERS
RTMLIB_DEVICE = 'cuda' if USE_CUDA else 'cpu'
ORT_PROVIDERS = ['CUDAExecutionProvider', 'CPUExecutionProvider'] if USE_CUDA else ['CPUExecutionProvider']

print(f"ONNX Runtime providers: {AVAILABLE_PROVIDERS} -> device='{RTMLIB_DEVICE}'")

# 1. 骨格検出モデルの初期化
wholebody = Wholebody(
    to_openpose=False,   # COCO WholeBody形式（133点）を使用
    mode='performance',  # 高精度なRTMW系モデルを使用
    backend='onnxruntime',
    device=RTMLIB_DEVICE,
)


class WD14Tagger:
    """
    SmilingWolf 系 WD14-Tagger (ONNX) を onnxruntime で直接推論するラッパー。
    タグの連続値信頼度スコア（0.0〜1.0）を取得できる。
    """

    def __init__(self, repo_id='SmilingWolf/wd-eva02-large-tagger-v3', providers=None):
        self.repo_id = repo_id
        self.providers = providers or ORT_PROVIDERS

        model_path = hf_hub_download(repo_id=repo_id, filename='model.onnx')
        tags_path = hf_hub_download(repo_id=repo_id, filename='selected_tags.csv')

        self.session = ort.InferenceSession(model_path, providers=self.providers)
        self.input_meta = self.session.get_inputs()[0]
        self.input_name = self.input_meta.name
        self.output_name = self.session.get_outputs()[0].name
        # 入力形状 [batch, height, width, 3] から正方形リサイズ解像度（通常448）を自動取得
        self.target_size = int(self.input_meta.shape[1])

        tags_df = pd.read_csv(tags_path)
        self.tag_names = tags_df['name'].astype(str).tolist()
        self.tag_categories = tags_df['category'].astype(int).tolist()
        self.rating_indices = [i for i, c in enumerate(self.tag_categories) if c == 9]
        self.general_indices = [i for i, c in enumerate(self.tag_categories) if c == 0]
        self.character_indices = [i for i, c in enumerate(self.tag_categories) if c == 4]

        print(f"Loaded WD14Tagger: {repo_id} (size={self.target_size}, tags={len(self.tag_names)})")

    def preprocess(self, img_bgr: np.ndarray) -> np.ndarray:
        """
        WD14-Tagger v3 標準の前処理:
        白背景で正方形パディング -> target_size へリサイズ -> BGR float32 (NHWC)。
        """
        if img_bgr.ndim == 2:
            img_bgr = cv2.cvtColor(img_bgr, cv2.COLOR_GRAY2BGR)
        elif img_bgr.shape[2] == 4:
            alpha = img_bgr[:, :, 3:4].astype(np.float32) / 255.0
            bgr = img_bgr[:, :, :3].astype(np.float32)
            white = np.full_like(bgr, 255.0)
            img_bgr = (bgr * alpha + white * (1.0 - alpha)).astype(np.uint8)

        h, w = img_bgr.shape[:2]
        max_dim = max(h, w)
        pad_top = (max_dim - h) // 2
        pad_bottom = max_dim - h - pad_top
        pad_left = (max_dim - w) // 2
        pad_right = max_dim - w - pad_left

        padded = cv2.copyMakeBorder(
            img_bgr,
            pad_top, pad_bottom, pad_left, pad_right,
            borderType=cv2.BORDER_CONSTANT,
            value=(255, 255, 255),
        )

        interp = cv2.INTER_AREA if max_dim > self.target_size else cv2.INTER_CUBIC
        resized = cv2.resize(padded, (self.target_size, self.target_size), interpolation=interp)
        # SmilingWolf ONNX モデルは BGR 順・[0, 255] スケールの float32 を期待する
        tensor = resized.astype(np.float32)
        return np.expand_dims(tensor, axis=0)

    def predict(self, img_bgr: np.ndarray) -> dict:
        """
        画像を推論し、rating / general / character の全信頼度スコア辞書を返す。
        """
        tensor = self.preprocess(img_bgr)
        probs = self.session.run([self.output_name], {self.input_name: tensor})[0][0]

        rating_scores = {self.tag_names[i]: float(probs[i]) for i in self.rating_indices}
        general_scores = {self.tag_names[i]: float(probs[i]) for i in self.general_indices}
        character_scores = {self.tag_names[i]: float(probs[i]) for i in self.character_indices}
        return {
            'rating': rating_scores,
            'general': general_scores,
            'character': character_scores,
        }


# 2. WD14-Tagger モデルの初期化
WD14_MODEL_REPO = 'SmilingWolf/wd-eva02-large-tagger-v3'
wd14_tagger = WD14Tagger(repo_id=WD14_MODEL_REPO, providers=ORT_PROVIDERS)

In [ ]:
# ==========================================
# アニメーション・立ち絵向けの可変スロット定義
# ==========================================
# ここに登録されたタグは「状態変数（表情・視線・構図・ポーズ・背景）」として
# 常に 0.0〜1.0 の連続値で記録され、キャラ固有の「不変特徴」自動抽出からは除外されます。
VARIABLE_TAG_SLOTS = {
    'expression': [
        'smile', 'grin', 'open_mouth', 'closed_mouth', 'parted_lips',
        'blush', 'expressionless', 'serious', 'angry', 'sad',
        'surprised', 'frown', 'embarrassed', 'nervous', 'tears',
        'closed_eyes', 'one_eye_closed', 'half-closed_eyes', 'wide-eyed',
    ],
    'gaze': [
        'looking_at_viewer', 'looking_away', 'looking_to_the_side',
        'looking_back', 'looking_up', 'looking_down',
        'facing_viewer', 'facing_away', 'profile',
        'from_side', 'from_behind', 'straight-on',
    ],
    'framing': [
        'portrait', 'close-up', 'upper_body', 'cowboy_shot',
        'full_body', 'wide_shot', 'dutch_angle', 'from_above', 'from_below',
    ],
    'pose_action': [
        'standing', 'sitting', 'walking', 'running', 'jumping',
        'arms_up', 'arm_up', 'hand_on_hip', 'hands_on_hips', 'crossed_arms',
        'peace_sign', 'waving', 'holding', 'leaning_forward', 'head_tilt',
        'hands_behind_back', 'arms_behind_back',
    ],
    'background': [
        'simple_background', 'white_background', 'grey_background',
        'black_background', 'blue_background', 'gradient_background',
        'transparent_background', 'outdoors', 'indoors', 'sky',
    ],
}

# キャラ固有特徴（不変タグ）の自動抽出から除外するメタ・人数系タグ
IGNORED_IDENTITY_TAGS = {
    '1girl', '1boy', 'solo', 'solo_focus', '2girls', '2boys', 'multiple_girls',
    'monochrome', 'greyscale', 'comic', 'highres', 'absurdres', 'commentary_request',
    'shadow', 'depth_of_field', 'blurry', 'blurry_background', 'motion_blur',
    'light_rays', 'lens_flare', 'vignetting',
}

ALL_VARIABLE_TAGS = {
    tag for tags in VARIABLE_TAG_SLOTS.values() for tag in tags
}
print(f"可変スロット数: {len(VARIABLE_TAG_SLOTS)} / 登録タグ数: {len(ALL_VARIABLE_TAGS)}")

## 4. 画像変数化・キャラクター分布構築ユーティリティ

以下の関数群を定義します。

- `extract_pose_variables`: COCO WholeBody（133点）から構図・向き・スケール不変プロポーション・関節角度・正規化骨格を抽出
- `extract_tag_variables`: WD14 の信頼度スコアをスロット別連続値変数＋高信頼度タグに構造化
- `extract_image_variables`: 1枚の画像を統合的に変数化（画像メタデータ＋骨格変数＋WD14変数）
- `build_character_profile`: 複数の LoRA 学習データから「キャラ固有特徴の統計分布（不変タグ分布＋骨格プロポーション分布＋可変スロット基準分布）」を構築
- `evaluate_against_profile`: 対象画像をキャラプロファイルと照合し、一貫性スコア（アイデンティティ・プロポーション）と逸脱アラートを算出
- `variables_to_flat_row`: 一覧比較・CSV出力向けにフラットな行辞書へ変換

In [ ]:
# COCO Body 17キーポイントのインデックス定義（WholeBody 133点の先頭17点と一致）
COCO_BODY_NAMES = [
    'nose', 'left_eye', 'right_eye', 'left_ear', 'right_ear',
    'left_shoulder', 'right_shoulder', 'left_elbow', 'right_elbow',
    'left_wrist', 'right_wrist', 'left_hip', 'right_hip',
    'left_knee', 'right_knee', 'left_ankle', 'right_ankle',
]
KP = {name: idx for idx, name in enumerate(COCO_BODY_NAMES)}


def _round_or_none(val, digits=4):
    if val is None or (isinstance(val, float) and (math.isnan(val) or math.isinf(val))):
        return None
    return round(float(val), digits)


def _select_primary_person(kpts, scores, kpt_thr=0.3):
    """複数人検出された場合、体幹・顔を含む有効点が多く面積が大きいメイン人物を選択する。"""
    if kpts is None or len(kpts) == 0:
        return None, None
    if len(kpts) == 1:
        return np.asarray(kpts[0], dtype=np.float32), np.asarray(scores[0], dtype=np.float32)

    best_idx = 0
    best_weight = -1.0
    for idx in range(len(kpts)):
        body_pts = np.asarray(kpts[idx][:17], dtype=np.float32)
        body_sc = np.asarray(scores[idx][:17], dtype=np.float32)
        valid = body_sc >= kpt_thr
        if not np.any(valid):
            continue
        pts = body_pts[valid]
        area = float((pts[:, 0].max() - pts[:, 0].min()) * (pts[:, 1].max() - pts[:, 1].min()))
        weight = float(body_sc[valid].mean()) * (area ** 0.5 + 1.0)
        if weight > best_weight:
            best_weight = weight
            best_idx = idx
    return np.asarray(kpts[best_idx], dtype=np.float32), np.asarray(scores[best_idx], dtype=np.float32)


def _point_angle_deg(a, b, c):
    """3点 A-B-C における頂点 B の内角（0〜180度）を計算する。"""
    v1 = a - b
    v2 = c - b
    n1 = np.linalg.norm(v1)
    n2 = np.linalg.norm(v2)
    if n1 < 1e-6 or n2 < 1e-6:
        return None
    cos_val = float(np.clip(np.dot(v1, v2) / (n1 * n2), -1.0, 1.0))
    return float(np.degrees(np.arccos(cos_val)))


def _line_tilt_deg(p_left, p_right):
    """画面左側の点から右側の点へのラインの水平からの傾き（度）を計算する。"""
    # 画像座標では通常 right_eye / right_shoulder の方が X 座標が小さい（正面時）
    dx = float(p_left[0] - p_right[0])
    dy = float(p_left[1] - p_right[1])
    if abs(dx) < 1e-6 and abs(dy) < 1e-6:
        return None
    return float(np.degrees(np.arctan2(dy, dx)))


def extract_pose_variables(kpts, scores, img_shape, kpt_thr=0.35):
    """
    rtmlib WholeBody の検出結果から、立ち絵・アニメーション向けの数値変数を抽出する。
    """
    h, w = img_shape[:2]
    p, s = _select_primary_person(kpts, scores, kpt_thr=kpt_thr)

    if p is None or s is None:
        return {
            'detection_quality': {
                'is_valid': False,
                'person_count': 0,
                'body_valid_ratio': 0.0,
                'body_mean_score': 0.0,
            },
            'framing': {},
            'orientation': {},
            'proportions': {},
            'joint_angles': {},
            'normalized_keypoints': {},
        }

    body_p = p[:17]
    body_s = s[:17]

    def has(*names):
        return all(body_s[KP[n]] >= kpt_thr for n in names)

    def pt(name):
        return body_p[KP[name]]

    valid_mask = body_s >= kpt_thr
    valid_pts = body_p[valid_mask]

    # --- 1. 画角・構図（framing） ---
    bbox_w_ratio = bbox_h_ratio = center_x = center_y = None
    if len(valid_pts) > 0:
        x_min, y_min = valid_pts[:, 0].min(), valid_pts[:, 1].min()
        x_max, y_max = valid_pts[:, 0].max(), valid_pts[:, 1].max()
        bbox_w_ratio = (x_max - x_min) / w
        bbox_h_ratio = (y_max - y_min) / h

    # 体幹重心（両肩＋両腰、なければ両肩、なければ有効点重心）
    if has('left_shoulder', 'right_shoulder', 'left_hip', 'right_hip'):
        core = np.mean([pt('left_shoulder'), pt('right_shoulder'), pt('left_hip'), pt('right_hip')], axis=0)
        center_x, center_y = core[0] / w, core[1] / h
    elif has('left_shoulder', 'right_shoulder'):
        core = np.mean([pt('left_shoulder'), pt('right_shoulder')], axis=0)
        center_x, center_y = core[0] / w, core[1] / h
    elif len(valid_pts) > 0:
        core = np.mean(valid_pts, axis=0)
        center_x, center_y = core[0] / w, core[1] / h

    shoulder_width = float(np.linalg.norm(pt('left_shoulder') - pt('right_shoulder'))) if has('left_shoulder', 'right_shoulder') else None
    eye_dist = float(np.linalg.norm(pt('left_eye') - pt('right_eye'))) if has('left_eye', 'right_eye') else None
    shoulder_width_ratio = (shoulder_width / w) if shoulder_width is not None else None
    eye_dist_ratio = (eye_dist / w) if eye_dist is not None else None

    # 検出された最下部部位からショット種別を推定
    has_ankles = has('left_ankle') or has('right_ankle')
    has_knees = has('left_knee') or has('right_knee')
    has_hips = has('left_hip') or has('right_hip')
    has_shoulders = has('left_shoulder') or has('right_shoulder')
    if has_ankles:
        estimated_shot = 'full_body'
    elif has_knees:
        estimated_shot = 'cowboy_shot'
    elif has_hips:
        estimated_shot = 'upper_body'
    elif has_shoulders:
        estimated_shot = 'bust_up'
    else:
        estimated_shot = 'close_up'

    # --- 2. 向き・傾き（orientation） ---
    head_roll_deg = _line_tilt_deg(pt('left_eye'), pt('right_eye')) if has('left_eye', 'right_eye') else None
    shoulder_tilt_deg = _line_tilt_deg(pt('left_shoulder'), pt('right_shoulder')) if has('left_shoulder', 'right_shoulder') else None

    # 顔の左右向き近似（Yaw proxy: 0=正面, 正/負=左右向き）
    head_yaw_proxy = None
    if has('nose', 'left_eye', 'right_eye') and eye_dist and eye_dist > 1e-3:
        eye_mid = 0.5 * (pt('left_eye') + pt('right_eye'))
        eye_vec = pt('left_eye') - pt('right_eye')
        eye_unit = eye_vec / np.linalg.norm(eye_vec)
        nose_offset = float(np.dot(pt('nose') - eye_mid, eye_unit))
        head_yaw_proxy = float(np.clip(nose_offset / (eye_dist * 0.5), -2.0, 2.0))

    # 体幹の傾き（垂直からのズレ角度、度）
    torso_tilt_deg = None
    torso_len = None
    if has('left_shoulder', 'right_shoulder', 'left_hip', 'right_hip'):
        sh_mid = 0.5 * (pt('left_shoulder') + pt('right_shoulder'))
        hip_mid = 0.5 * (pt('left_hip') + pt('right_hip'))
        torso_vec = hip_mid - sh_mid
        torso_len = float(np.linalg.norm(torso_vec))
        if torso_len > 1e-3:
            # 下向き(0, 1)を 0度 とした左右傾き
            torso_tilt_deg = float(np.degrees(np.arctan2(torso_vec[0], torso_vec[1])))

    if head_yaw_proxy is None:
        facing_direction = 'unknown'
    elif abs(head_yaw_proxy) < 0.25:
        facing_direction = 'front'
    elif abs(head_yaw_proxy) < 0.75:
        facing_direction = 'three_quarter_right' if head_yaw_proxy > 0 else 'three_quarter_left'
    else:
        facing_direction = 'side_right' if head_yaw_proxy > 0 else 'side_left'

    # --- 3. スケール不変プロポーション（proportions: キャラ体型一貫性チェック用） ---
    eye_to_shoulder_ratio = None
    torso_to_shoulder_ratio = None
    hip_to_shoulder_ratio = None
    upper_arm_to_shoulder_ratio = None
    if shoulder_width and shoulder_width > 1e-3:
        if eye_dist is not None:
            eye_to_shoulder_ratio = eye_dist / shoulder_width
        if torso_len is not None:
            torso_to_shoulder_ratio = torso_len / shoulder_width
        if has('left_hip', 'right_hip'):
            hip_width = float(np.linalg.norm(pt('left_hip') - pt('right_hip')))
            hip_to_shoulder_ratio = hip_width / shoulder_width
        arm_lens = []
        if has('left_shoulder', 'left_elbow'):
            arm_lens.append(float(np.linalg.norm(pt('left_elbow') - pt('left_shoulder'))))
        if has('right_shoulder', 'right_elbow'):
            arm_lens.append(float(np.linalg.norm(pt('right_elbow') - pt('right_shoulder'))))
        if arm_lens:
            upper_arm_to_shoulder_ratio = float(np.mean(arm_lens)) / shoulder_width

    # --- 4. 関節角度（joint_angles） ---
    left_elbow_deg = _point_angle_deg(pt('left_shoulder'), pt('left_elbow'), pt('left_wrist')) if has('left_shoulder', 'left_elbow', 'left_wrist') else None
    right_elbow_deg = _point_angle_deg(pt('right_shoulder'), pt('right_elbow'), pt('right_wrist')) if has('right_shoulder', 'right_elbow', 'right_wrist') else None
    left_knee_deg = _point_angle_deg(pt('left_hip'), pt('left_knee'), pt('left_ankle')) if has('left_hip', 'left_knee', 'left_ankle') else None
    right_knee_deg = _point_angle_deg(pt('right_hip'), pt('right_knee'), pt('right_ankle')) if has('right_hip', 'right_knee', 'right_ankle') else None

    # 上腕の挙上角度（下向き体幹軸を 0度、真上を 180度 とする）
    down_ref = np.array([0.0, 1.0], dtype=np.float32)
    if has('left_shoulder', 'right_shoulder', 'left_hip', 'right_hip') and torso_len and torso_len > 1e-3:
        down_ref = (0.5 * (pt('left_hip') + pt('right_hip')) - 0.5 * (pt('left_shoulder') + pt('right_shoulder'))) / torso_len

    def _arm_elev(sh_name, el_name):
        if not has(sh_name, el_name):
            return None
        v = pt(el_name) - pt(sh_name)
        n = np.linalg.norm(v)
        if n < 1e-6:
            return None
        return float(np.degrees(np.arccos(np.clip(np.dot(v / n, down_ref), -1.0, 1.0))))

    left_shoulder_elev_deg = _arm_elev('left_shoulder', 'left_elbow')
    right_shoulder_elev_deg = _arm_elev('right_shoulder', 'right_elbow')

    # --- 5. 正規化キーポイント（肩中点原点・肩幅=1.0、ポーズ類似度・補間用） ---
    normalized_keypoints = {}
    if has('left_shoulder', 'right_shoulder') and shoulder_width and shoulder_width > 1e-3:
        origin = 0.5 * (pt('left_shoulder') + pt('right_shoulder'))
        for idx, name in enumerate(COCO_BODY_NAMES):
            rel = (body_p[idx] - origin) / shoulder_width
            normalized_keypoints[name] = {
                'x': _round_or_none(rel[0], 4),
                'y': _round_or_none(rel[1], 4),
                'score': _round_or_none(body_s[idx], 4),
            }

    return {
        'detection_quality': {
            'is_valid': bool(np.sum(valid_mask) >= 4),
            'person_count': int(len(kpts)),
            'body_valid_ratio': _round_or_none(float(np.mean(valid_mask)), 4),
            'body_mean_score': _round_or_none(float(np.mean(body_s)), 4),
        },
        'framing': {
            'estimated_shot': estimated_shot,
            'bbox_w_ratio': _round_or_none(bbox_w_ratio),
            'bbox_h_ratio': _round_or_none(bbox_h_ratio),
            'center_x': _round_or_none(center_x),
            'center_y': _round_or_none(center_y),
            'shoulder_width_ratio': _round_or_none(shoulder_width_ratio),
            'eye_dist_ratio': _round_or_none(eye_dist_ratio),
        },
        'orientation': {
            'facing_direction': facing_direction,
            'head_yaw_proxy': _round_or_none(head_yaw_proxy),
            'head_roll_deg': _round_or_none(head_roll_deg, 2),
            'shoulder_tilt_deg': _round_or_none(shoulder_tilt_deg, 2),
            'torso_tilt_deg': _round_or_none(torso_tilt_deg, 2),
        },
        'proportions': {
            'eye_to_shoulder_ratio': _round_or_none(eye_to_shoulder_ratio),
            'torso_to_shoulder_ratio': _round_or_none(torso_to_shoulder_ratio),
            'hip_to_shoulder_ratio': _round_or_none(hip_to_shoulder_ratio),
            'upper_arm_to_shoulder_ratio': _round_or_none(upper_arm_to_shoulder_ratio),
        },
        'joint_angles': {
            'left_elbow_deg': _round_or_none(left_elbow_deg, 2),
            'right_elbow_deg': _round_or_none(right_elbow_deg, 2),
            'left_shoulder_elev_deg': _round_or_none(left_shoulder_elev_deg, 2),
            'right_shoulder_elev_deg': _round_or_none(right_shoulder_elev_deg, 2),
            'left_knee_deg': _round_or_none(left_knee_deg, 2),
            'right_knee_deg': _round_or_none(right_knee_deg, 2),
        },
        'normalized_keypoints': normalized_keypoints,
    }

In [ ]:
def extract_tag_variables(
    wd14_pred: dict,
    slot_definitions: dict = VARIABLE_TAG_SLOTS,
    general_threshold: float = 0.35,
    character_threshold: float = 0.50,
):
    """
    WD14-Tagger の推論結果から、カテゴリ別の連続値スロット変数と高信頼度タグを抽出する。
    """
    general_scores = wd14_pred['general']
    rating_scores = wd14_pred['rating']
    character_scores = wd14_pred['character']

    # スロット別連続値変数（0.0〜1.0）
    slots = {}
    slot_dominant = {}
    for slot_name, tag_list in slot_definitions.items():
        slot_dict = {
            tag: round(float(general_scores.get(tag, 0.0)), 4)
            for tag in tag_list
        }
        slots[slot_name] = slot_dict
        if slot_dict:
            top_tag, top_score = max(slot_dict.items(), key=lambda kv: kv[1])
            slot_dominant[slot_name] = {'tag': top_tag, 'score': top_score}

    active_general = {
        tag: round(float(score), 4)
        for tag, score in sorted(general_scores.items(), key=lambda kv: kv[1], reverse=True)
        if score >= general_threshold
    }
    active_character = {
        tag: round(float(score), 4)
        for tag, score in sorted(character_scores.items(), key=lambda kv: kv[1], reverse=True)
        if score >= character_threshold
    }

    return {
        'slots': slots,
        'slot_dominant': slot_dominant,
        'rating': {k: round(float(v), 4) for k, v in rating_scores.items()},
        'active_general_tags': active_general,
        'active_character_tags': active_character,
    }


def extract_image_variables(
    img_path: str,
    pose_model=None,
    tagger=None,
    slot_definitions: dict = VARIABLE_TAG_SLOTS,
    kpt_thr: float = 0.35,
    general_threshold: float = 0.35,
    return_debug_artifacts: bool = False,
):
    """
    1枚の画像を読み込み、Pose（骨格）と WD14-Tagger（タグ信頼度）の両面から変数化する。
    """
    pose_model = pose_model or wholebody
    tagger = tagger or wd14_tagger

    img_bgr = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
    assert img_bgr is not None, f"画像が読み込めません: {img_path}"

    # アルファチャンネル付き立ち絵 PNG の場合は白背景合成した BGR を骨格検出へ渡す
    if img_bgr.ndim == 2:
        bgr_for_pose = cv2.cvtColor(img_bgr, cv2.COLOR_GRAY2BGR)
    elif img_bgr.shape[2] == 4:
        alpha = img_bgr[:, :, 3:4].astype(np.float32) / 255.0
        bgr = img_bgr[:, :, :3].astype(np.float32)
        bgr_for_pose = (bgr * alpha + 255.0 * (1.0 - alpha)).astype(np.uint8)
    else:
        bgr_for_pose = img_bgr

    h, w = bgr_for_pose.shape[:2]

    # 1. 骨格変数
    kpts, scores = pose_model(bgr_for_pose)
    pose_vars = extract_pose_variables(kpts, scores, bgr_for_pose.shape, kpt_thr=kpt_thr)

    # 2. WD14-Tagger 変数
    wd14_pred = tagger.predict(img_bgr)
    tag_vars = extract_tag_variables(
        wd14_pred,
        slot_definitions=slot_definitions,
        general_threshold=general_threshold,
    )

    record = {
        'image_id': os.path.splitext(os.path.basename(img_path))[0],
        'file_name': os.path.basename(img_path),
        'file_path': img_path,
        'meta': {
            'width': int(w),
            'height': int(h),
            'aspect_ratio': round(w / h, 4),
            'has_alpha': bool(img_bgr.ndim == 3 and img_bgr.shape[2] == 4),
        },
        'pose': pose_vars,
        'tags': tag_vars,
    }

    if return_debug_artifacts:
        debug_skel = draw_skeleton(bgr_for_pose.copy(), kpts, scores, kpt_thr=kpt_thr)
        return record, wd14_pred, bgr_for_pose, debug_skel
    return record, wd14_pred

In [ ]:
def _summarize_numeric_distribution(values):
    """数値リストから平均・標準偏差・中央値・最小・最大・件数を集計する。"""
    clean = [float(v) for v in values if v is not None and not math.isnan(float(v))]
    if not clean:
        return None
    arr = np.asarray(clean, dtype=np.float64)
    return {
        'count': int(len(arr)),
        'mean': round(float(np.mean(arr)), 4),
        'std': round(float(np.std(arr, ddof=0)), 4),
        'median': round(float(np.median(arr)), 4),
        'min': round(float(np.min(arr)), 4),
        'max': round(float(np.max(arr)), 4),
    }


def list_image_files(directory: str, extensions=('.png', '.jpg', '.jpeg', '.webp')):
    """指定ディレクトリ内の画像ファイルをソート済みパス一覧で返す。"""
    if not os.path.isdir(directory):
        return []
    files = [
        os.path.join(directory, f)
        for f in sorted(os.listdir(directory))
        if os.path.isfile(os.path.join(directory, f)) and f.lower().endswith(extensions)
    ]
    return files


def build_character_profile(
    reference_paths: list,
    character_name: str = 'character_default',
    pose_model=None,
    tagger=None,
    slot_definitions: dict = VARIABLE_TAG_SLOTS,
    ignored_identity_tags: set = IGNORED_IDENTITY_TAGS,
    force_identity_tags: list = None,
    exclude_identity_tags: list = None,
    core_presence_threshold: float = 0.65,
    core_mean_threshold: float = 0.45,
    tag_active_threshold: float = 0.35,
    kpt_thr: float = 0.35,
):
    """
    複数の LoRA 学習データ／リファレンス画像群を解析し、
    キャラクター固有の特徴分布（Character Profile）を構築する。

    - identity_core_tags: 出現率・平均信頼度が高い「キャラ固有の不変特徴タグ」の分布
    - background_tag_stats: 外乱タグ混入チェック用の全一般タグの平均信頼度（上位・出現分）
    - proportions: スケール不変な骨格プロポーション比率（頭身・肩幅比など）の分布
    - variable_slots_baseline: 表情・視線・構図などの可変スロットの基準分布
    """
    assert len(reference_paths) > 0, "リファレンス画像が1枚も見つかりません。"

    force_set = set(force_identity_tags or [])
    exclude_set = set(exclude_identity_tags or [])
    variable_tag_set = {t for tags in slot_definitions.values() for t in tags}

    records = []
    all_general_preds = []

    for idx, path in enumerate(reference_paths, 1):
        rec, wd14_pred = extract_image_variables(
            path,
            pose_model=pose_model,
            tagger=tagger,
            slot_definitions=slot_definitions,
            kpt_thr=kpt_thr,
            general_threshold=tag_active_threshold,
        )
        records.append(rec)
        all_general_preds.append(wd14_pred['general'])
        print(f"[Profile {idx}/{len(reference_paths)}] {rec['file_name']} "
              f"(pose_valid={rec['pose']['detection_quality']['is_valid']}, "
              f"shot={rec['pose']['framing'].get('estimated_shot')})")

    n_imgs = len(records)
    all_tag_names = list(all_general_preds[0].keys())

    # 1. キャラ固有の不変特徴タグ（identity_core_tags）の統計分布を自動同定
    identity_core_tags = {}
    reference_general_mean = {}

    for tag in all_tag_names:
        vals = np.array([pred[tag] for pred in all_general_preds], dtype=np.float64)
        mean_val = float(np.mean(vals))
        max_val = float(np.max(vals))
        if max_val >= 0.15:
            reference_general_mean[tag] = round(mean_val, 4)

        if tag in exclude_set:
            continue
        is_forced = tag in force_set
        if not is_forced and (tag in variable_tag_set or tag in ignored_identity_tags):
            continue

        presence_rate = float(np.mean(vals >= tag_active_threshold))
        if is_forced or (presence_rate >= core_presence_threshold and mean_val >= core_mean_threshold):
            identity_core_tags[tag] = {
                'mean': round(mean_val, 4),
                'std': round(float(np.std(vals, ddof=0)), 4),
                'median': round(float(np.median(vals)), 4),
                'min': round(float(np.min(vals)), 4),
                'max': round(float(np.max(vals)), 4),
                'presence_rate': round(presence_rate, 4),
                'forced': is_forced,
            }

    # 平均信頼度順にソート
    identity_core_tags = dict(
        sorted(identity_core_tags.items(), key=lambda kv: kv[1]['mean'], reverse=True)
    )

    # 2. スケール不変骨格プロポーションの分布
    proportion_keys = [
        'eye_to_shoulder_ratio',
        'torso_to_shoulder_ratio',
        'hip_to_shoulder_ratio',
        'upper_arm_to_shoulder_ratio',
    ]
    proportions_dist = {}
    for key in proportion_keys:
        vals = [r['pose']['proportions'].get(key) for r in records if r['pose']['detection_quality']['is_valid']]
        summary = _summarize_numeric_distribution(vals)
        if summary is not None:
            proportions_dist[key] = summary

    # 3. 可変スロットのベースライン分布
    variable_slots_baseline = {}
    for slot_name, tag_list in slot_definitions.items():
        slot_stats = {}
        for tag in tag_list:
            vals = [r['tags']['slots'][slot_name][tag] for r in records]
            slot_stats[tag] = _summarize_numeric_distribution(vals)
        variable_slots_baseline[slot_name] = slot_stats

    # 4. 構図・向きのベースライン分布
    framing_baseline = {}
    for key in ['bbox_w_ratio', 'bbox_h_ratio', 'center_x', 'center_y', 'shoulder_width_ratio', 'eye_dist_ratio']:
        vals = [r['pose']['framing'].get(key) for r in records if r['pose']['detection_quality']['is_valid']]
        summary = _summarize_numeric_distribution(vals)
        if summary is not None:
            framing_baseline[key] = summary

    profile = {
        'schema_version': '1.0',
        'character_name': character_name,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'wd14_model': getattr(tagger or wd14_tagger, 'repo_id', 'unknown'),
        'reference_count': n_imgs,
        'reference_files': [r['file_name'] for r in records],
        'thresholds': {
            'core_presence_threshold': core_presence_threshold,
            'core_mean_threshold': core_mean_threshold,
            'tag_active_threshold': tag_active_threshold,
            'kpt_thr': kpt_thr,
        },
        'identity_core_tags': identity_core_tags,
        'proportions': proportions_dist,
        'framing_baseline': framing_baseline,
        'variable_slots_baseline': variable_slots_baseline,
        'reference_general_mean': reference_general_mean,
    }
    return profile, records


def evaluate_against_profile(
    record: dict,
    wd14_pred: dict,
    profile: dict,
    slot_definitions: dict = VARIABLE_TAG_SLOTS,
    ignored_identity_tags: set = IGNORED_IDENTITY_TAGS,
    identity_z_warn: float = 2.0,
    identity_min_std: float = 0.08,
    proportion_z_warn: float = 2.5,
    proportion_min_std: float = 0.04,
    unexpected_tag_threshold: float = 0.55,
    unexpected_ref_max_mean: float = 0.15,
):
    """
    対象画像の変数化結果をキャラクター固有分布（Character Profile）と照合し、
    一貫性スコアおよび逸脱アラートを算出する。
    """
    general_scores = wd14_pred['general']
    core_tags = profile.get('identity_core_tags', {})
    ref_general_mean = profile.get('reference_general_mean', {})
    variable_tag_set = {t for tags in slot_definitions.values() for t in tags}

    # --- 1. アイデンティティ一貫性（不変タグ分布との比較） ---
    core_details = {}
    missing_or_weak_traits = []
    weighted_match_sum = 0.0
    weight_total = 0.0

    for tag, stats in core_tags.items():
        actual = float(general_scores.get(tag, 0.0))
        mean = float(stats['mean'])
        std = float(stats['std'])
        eff_std = max(std, identity_min_std)
        z_score = (actual - mean) / eff_std

        # 平均より大きく下回った場合のみペナルティ（上回るのは特徴が強く出ているためOK）
        shortfall = max(0.0, mean - actual)
        match = max(0.0, 1.0 - shortfall / max(mean, 1e-6))
        weighted_match_sum += mean * match
        weight_total += mean

        is_alert = (z_score <= -identity_z_warn) and (shortfall >= 0.20)
        core_details[tag] = {
            'actual': round(actual, 4),
            'expected_mean': round(mean, 4),
            'expected_std': round(std, 4),
            'z_score': round(z_score, 2),
            'alert': is_alert,
        }
        if is_alert:
            missing_or_weak_traits.append({
                'tag': tag,
                'actual': round(actual, 4),
                'expected_mean': round(mean, 4),
                'expected_std': round(std, 4),
                'z_score': round(z_score, 2),
            })

    identity_score = (weighted_match_sum / weight_total) if weight_total > 0 else None

    # 学習データでほとんど出ないのに今回強く出た非可変タグ（衣装崩れ・余計なアクセサリ混入など）を検知
    unexpected_traits = []
    for tag, score in sorted(general_scores.items(), key=lambda kv: kv[1], reverse=True):
        if score < unexpected_tag_threshold:
            break
        if tag in core_tags or tag in variable_tag_set or tag in ignored_identity_tags:
            continue
        ref_mean = float(ref_general_mean.get(tag, 0.0))
        if ref_mean <= unexpected_ref_max_mean:
            unexpected_traits.append({
                'tag': tag,
                'actual': round(float(score), 4),
                'reference_mean': round(ref_mean, 4),
            })

    # --- 2. 骨格プロポーション一貫性（頭身・肩幅比の分布比較） ---
    prop_profile = profile.get('proportions', {})
    prop_actual = record['pose'].get('proportions', {})
    prop_details = {}
    prop_alerts = []
    prop_scores = []

    for key, stats in prop_profile.items():
        actual = prop_actual.get(key)
        if actual is None:
            continue
        mean = float(stats['mean'])
        std = float(stats['std'])
        eff_std = max(std, proportion_min_std)
        z_score = (float(actual) - mean) / eff_std
        is_alert = abs(z_score) >= proportion_z_warn
        # |z| が 0 なら 1.0、z_warn 以上で大きく低下する滑らかな一致スコア
        prop_scores.append(float(math.exp(-0.5 * (z_score / proportion_z_warn) ** 2)))
        prop_details[key] = {
            'actual': round(float(actual), 4),
            'expected_mean': round(mean, 4),
            'expected_std': round(std, 4),
            'z_score': round(z_score, 2),
            'alert': is_alert,
        }
        if is_alert:
            prop_alerts.append({
                'metric': key,
                'actual': round(float(actual), 4),
                'expected_mean': round(mean, 4),
                'z_score': round(z_score, 2),
            })

    proportion_score = float(np.mean(prop_scores)) if prop_scores else None

    # --- 3. 可変スロットのベースライン差分（表情・視線・構図がどれだけ動いたか） ---
    slot_deltas = {}
    baseline_slots = profile.get('variable_slots_baseline', {})
    for slot_name, slot_vals in record['tags']['slots'].items():
        base = baseline_slots.get(slot_name, {})
        deltas = {}
        for tag, val in slot_vals.items():
            base_mean = base.get(tag, {}).get('mean', 0.0) if base.get(tag) else 0.0
            deltas[tag] = round(float(val) - float(base_mean), 4)
        slot_deltas[slot_name] = deltas

    # 総合スコア（アイデンティティ 70% + プロポーション 30%）
    if identity_score is not None and proportion_score is not None:
        overall_score = 0.7 * identity_score + 0.3 * proportion_score
    else:
        overall_score = identity_score if identity_score is not None else proportion_score

    return {
        'character_name': profile.get('character_name'),
        'overall_consistency': _round_or_none(overall_score),
        'identity_consistency': _round_or_none(identity_score),
        'proportion_consistency': _round_or_none(proportion_score),
        'missing_or_weak_traits': missing_or_weak_traits,
        'unexpected_traits': unexpected_traits[:10],
        'proportion_alerts': prop_alerts,
        'core_tag_comparison': core_details,
        'proportion_comparison': prop_details,
        'variable_slot_deltas': slot_deltas,
    }


def variables_to_flat_row(record: dict, consistency: dict = None) -> dict:
    """
    一覧表（DataFrame / CSV）用に、画像変数と一貫性評価を1行のフラット辞書へ整形する。
    """
    pose = record['pose']
    framing = pose.get('framing', {})
    orient = pose.get('orientation', {})
    props = pose.get('proportions', {})
    angles = pose.get('joint_angles', {})
    dom = record['tags'].get('slot_dominant', {})

    row = {
        'image_id': record['image_id'],
        'file_name': record['file_name'],
        'width': record['meta']['width'],
        'height': record['meta']['height'],
        'pose_valid': pose['detection_quality']['is_valid'],
        'shot_pose': framing.get('estimated_shot'),
        'shot_tag': dom.get('framing', {}).get('tag'),
        'facing': orient.get('facing_direction'),
        'head_yaw': orient.get('head_yaw_proxy'),
        'head_roll_deg': orient.get('head_roll_deg'),
        'shoulder_tilt_deg': orient.get('shoulder_tilt_deg'),
        'center_x': framing.get('center_x'),
        'center_y': framing.get('center_y'),
        'eye_to_shoulder': props.get('eye_to_shoulder_ratio'),
        'torso_to_shoulder': props.get('torso_to_shoulder_ratio'),
        'l_elbow_deg': angles.get('left_elbow_deg'),
        'r_elbow_deg': angles.get('right_elbow_deg'),
        'top_expression': dom.get('expression', {}).get('tag'),
        'top_expression_score': dom.get('expression', {}).get('score'),
        'top_gaze': dom.get('gaze', {}).get('tag'),
        'top_gaze_score': dom.get('gaze', {}).get('score'),
        'top_pose_tag': dom.get('pose_action', {}).get('tag'),
        'top_pose_score': dom.get('pose_action', {}).get('score'),
    }

    if consistency is not None:
        row.update({
            'overall_consistency': consistency.get('overall_consistency'),
            'identity_consistency': consistency.get('identity_consistency'),
            'proportion_consistency': consistency.get('proportion_consistency'),
            'weak_trait_count': len(consistency.get('missing_or_weak_traits', [])),
            'unexpected_trait_count': len(consistency.get('unexpected_traits', [])),
            'prop_alert_count': len(consistency.get('proportion_alerts', [])),
        })
    return row

In [ ]:
def _style_image_axis(ax, title, border_color='gray'):
    """画像表示用の軸から目盛りを消し、枠線を表示する。"""
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)
    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_edgecolor(border_color)
        spine.set_linewidth(1.5)


def show_profile_summary(profile: dict, top_n: int = 20, figsize=(14, 6)):
    """
    構築したキャラクター固有分布（不変タグの平均±標準偏差 & プロポーション分布）を可視化する。
    """
    core_tags = profile.get('identity_core_tags', {})
    props = profile.get('proportions', {})

    fig, axes = plt.subplots(1, 2, figsize=figsize, gridspec_kw={'width_ratios': [2.2, 1.0]})

    # 左: キャラ固有不変タグ（上位 top_n）
    ax_tags = axes[0]
    items = list(core_tags.items())[:top_n]
    if items:
        names = [k for k, _ in items][::-1]
        means = [v['mean'] for _, v in items][::-1]
        stds = [v['std'] for _, v in items][::-1]
        y_pos = np.arange(len(names))
        ax_tags.barh(y_pos, means, xerr=stds, color='#4C78A8', alpha=0.85, ecolor='#333333', capsize=3)
        ax_tags.set_yticks(y_pos)
        ax_tags.set_yticklabels(names, fontsize=9)
        ax_tags.set_xlim(0.0, 1.08)
        ax_tags.set_xlabel('WD14 Confidence (Mean +/- Std)')
        ax_tags.set_title(f"Identity Core Tags (n={profile['reference_count']}, total={len(core_tags)})")
        ax_tags.grid(axis='x', linestyle='--', alpha=0.4)
    else:
        ax_tags.text(0.5, 0.5, 'No Identity Core Tags Found', ha='center', va='center')
        ax_tags.set_axis_off()

    # 右: 骨格プロポーション分布
    ax_prop = axes[1]
    if props:
        p_names = list(props.keys())[::-1]
        p_means = [props[k]['mean'] for k in p_names]
        p_stds = [props[k]['std'] for k in p_names]
        y_pos = np.arange(len(p_names))
        ax_prop.barh(y_pos, p_means, xerr=p_stds, color='#72B7B2', alpha=0.85, ecolor='#333333', capsize=3)
        ax_prop.set_yticks(y_pos)
        ax_prop.set_yticklabels(p_names, fontsize=9)
        ax_prop.set_xlabel('Scale-Invariant Ratio (Mean +/- Std)')
        ax_prop.set_title('Pose Proportions Distribution')
        ax_prop.grid(axis='x', linestyle='--', alpha=0.4)
    else:
        ax_prop.text(0.5, 0.5, 'No Pose Proportions Available', ha='center', va='center')
        ax_prop.set_axis_off()

    plt.tight_layout()
    plt.show()


def show_target_report(record: dict, consistency: dict, debug_skel: np.ndarray, figsize=(15, 5)):
    """
    対象画像1枚について、骨格プレビュー・表情/視線スコア・キャラ固有タグの一致状況を並べて表示する。
    """
    fig, axes = plt.subplots(1, 3, figsize=figsize)

    # 1. 骨格検出プレビュー
    axes[0].imshow(cv2.cvtColor(debug_skel, cv2.COLOR_BGR2RGB))
    ov = consistency.get('overall_consistency')
    ov_str = f"{ov:.3f}" if ov is not None else "N/A"
    _style_image_axis(axes[0], f"{record['file_name']} (Consistency: {ov_str})")

    # 2. 表情・視線・構図の上位スコア
    ax_var = axes[1]
    slot_scores = []
    for slot_name in ['expression', 'gaze', 'framing', 'pose_action']:
        for tag, val in record['tags']['slots'].get(slot_name, {}).items():
            if val >= 0.15:
                slot_scores.append((f"[{slot_name[:4]}] {tag}", val))
    slot_scores = sorted(slot_scores, key=lambda x: x[1], reverse=True)[:12]
    if slot_scores:
        labels = [k for k, _ in slot_scores][::-1]
        vals = [v for _, v in slot_scores][::-1]
        ax_var.barh(np.arange(len(labels)), vals, color='#F58518', alpha=0.85)
        ax_var.set_yticks(np.arange(len(labels)))
        ax_var.set_yticklabels(labels, fontsize=8.5)
        ax_var.set_xlim(0.0, 1.05)
        ax_var.set_xlabel('Confidence')
        ax_var.set_title('Active Variable Slots (>= 0.15)')
        ax_var.grid(axis='x', linestyle='--', alpha=0.4)
    else:
        ax_var.text(0.5, 0.5, 'No Active Slot Tags', ha='center', va='center')
        ax_var.set_axis_off()

    # 3. キャラ固有不変タグの期待値 vs 実測値（ズレが大きい順に表示）
    ax_id = axes[2]
    comp = consistency.get('core_tag_comparison', {})
    if comp:
        ranked = sorted(comp.items(), key=lambda kv: kv[1]['z_score'])[:12]
        labels = [k for k, _ in ranked][::-1]
        actuals = [v['actual'] for _, v in ranked][::-1]
        expected = [v['expected_mean'] for _, v in ranked][::-1]
        colors = ['#E45756' if v['alert'] else '#54A24B' for _, v in ranked][::-1]
        y_pos = np.arange(len(labels))
        ax_id.barh(y_pos, actuals, color=colors, alpha=0.85, label='Actual')
        ax_id.scatter(expected, y_pos, color='#222222', zorder=5, s=30, label='Ref Mean')
        ax_id.set_yticks(y_pos)
        ax_id.set_yticklabels(labels, fontsize=8.5)
        ax_id.set_xlim(0.0, 1.08)
        ax_id.set_xlabel('Confidence (Red = Weak/Missing Alert)')
        ax_id.set_title('Identity Core Check (Lowest z-score)')
        ax_id.legend(loc='lower right', fontsize=8)
        ax_id.grid(axis='x', linestyle='--', alpha=0.4)
    else:
        ax_id.text(0.5, 0.5, 'No Core Tag Comparison', ha='center', va='center')
        ax_id.set_axis_off()

    plt.tight_layout()
    plt.show()

    # テキストでのアラートサマリー
    weak = consistency.get('missing_or_weak_traits', [])
    unexp = consistency.get('unexpected_traits', [])
    p_alerts = consistency.get('proportion_alerts', [])
    if weak:
        print("  ⚠️ 固有特徴の低下・欠落:", ", ".join(
            f"{w['tag']} (実測 {w['actual']:.2f} vs 基準 {w['expected_mean']:.2f}±{w['expected_std']:.2f})"
            for w in weak
        ))
    if unexp:
        print("  ⚠️ 学習分布にない外乱タグ:", ", ".join(
            f"{u['tag']} ({u['actual']:.2f}, 基準平均 {u['reference_mean']:.2f})"
            for u in unexp[:5]
        ))
    if p_alerts:
        print("  ⚠️ 骨格プロポーション逸脱:", ", ".join(
            f"{p['metric']} (実測 {p['actual']:.3f} vs 基準 {p['expected_mean']:.3f}, z={p['z_score']:+.2f})"
            for p in p_alerts
        ))
    if not weak and not unexp and not p_alerts:
        print("  ✅ キャラクター固有分布に対する大きな逸脱は検出されませんでした。")


def show_batch_variable_heatmap(records: list, slot_name: str = 'expression', min_max_score: float = 0.20, figsize=(12, 5)):
    """
    複数画像（立ち絵差分・アニメ連番）における指定スロットの信頼度スコア推移をヒートマップで表示する。
    """
    if len(records) < 2:
        print("ℹ️ ヒートマップ比較には対象画像が2枚以上必要です。")
        return

    slot_tags = list(records[0]['tags']['slots'].get(slot_name, {}).keys())
    matrix = np.array([
        [r['tags']['slots'][slot_name][t] for t in slot_tags]
        for r in records
    ], dtype=np.float32)

    # いずれかの画像で min_max_score 以上になったタグのみに絞って見やすくする
    keep = np.max(matrix, axis=0) >= min_max_score
    if not np.any(keep):
        print(f"ℹ️ スロット '{slot_name}' にしきい値 {min_max_score} 以上のタグがありません。")
        return

    filtered_tags = [t for t, k in zip(slot_tags, keep) if k]
    filtered_mat = matrix[:, keep]
    img_labels = [r['image_id'] for r in records]

    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(filtered_mat, aspect='auto', cmap='YlOrRd', vmin=0.0, vmax=1.0)
    ax.set_xticks(np.arange(len(filtered_tags)))
    ax.set_xticklabels(filtered_tags, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(np.arange(len(img_labels)))
    ax.set_yticklabels(img_labels, fontsize=9)
    ax.set_title(f"Batch Variable Heatmap: {slot_name}")
    fig.colorbar(im, ax=ax, label='WD14 Confidence')
    plt.tight_layout()
    plt.show()


def save_results_to_drive(output_dir: str, file_paths: list, prefix: str = ''):
    """生成した JSON / CSV / デバッグ画像を Google Drive へ保存する。"""
    if not os.path.exists('/content/drive'):
        print("⚠️ 警告: '/content/drive' が見つかりません。先に Drive をマウントしてください。")
        return

    os.makedirs(output_dir, exist_ok=True)
    saved_count = 0
    for src_path in file_paths:
        if not os.path.exists(src_path):
            print(f"ℹ️ スキップ（未生成）: {src_path}")
            continue
        dst_name = f"{prefix}{os.path.basename(src_path)}"
        dst_path = os.path.join(output_dir, dst_name)
        shutil.copy2(src_path, dst_path)
        print(f"✅ 保存完了: {dst_path}")
        saved_count += 1
    print(f"合計 {saved_count} ファイルを Drive に保存しました -> {output_dir}")

## 5. キャラクター固有分布（Character Profile）の構築

複数の LoRA 学習データ／リファレンス画像群（`REFERENCE_DIR`）を一括解析し、そのキャラクター固有の特徴分布を `character_profile.json` として構築・保存します。

- 既に作成済みのプロファイルを再利用したい場合は、`LOAD_EXISTING_PROFILE_PATH` に JSON パスを指定してください。
- `FORCE_IDENTITY_TAGS` / `EXCLUDE_IDENTITY_TAGS` を使うと、自動抽出された不変特徴タグをプロジェクト要件に合わせて微調整できます。

In [ ]:
# ==========================================
# パラメータ設定（Character Profile 構築）
# ==========================================
CHARACTER_NAME = 'my_character'
REFERENCE_DIR = './data/reference_set'
PROFILE_JSON_PATH = './character_profile.json'

# 既存プロファイルを読み込む場合はパスを指定（新規作成時は None）
LOAD_EXISTING_PROFILE_PATH = None

# キャラ固有不変タグの自動抽出基準
CORE_PRESENCE_THRESHOLD = 0.65   # 全リファレンス画像のうち何割以上で検出されたタグを固有特徴とみなすか
CORE_MEAN_THRESHOLD = 0.45       # 平均信頼度がこの値以上のタグを固有特徴とみなすか
TAG_ACTIVE_THRESHOLD = 0.35      # タグが「出現した」と判定する信頼度しきい値
KPT_THR = 0.35                   # 骨格キーポイントの有効しきい値

# 手動で必ず含めたい / 除外したいタグ（任意）
FORCE_IDENTITY_TAGS = []         # 例: ['hair_ornament', 'mole_under_eye']
EXCLUDE_IDENTITY_TAGS = []       # 例: ['thighhighs']

# ==========================================
# 実行
# ==========================================
if LOAD_EXISTING_PROFILE_PATH and os.path.exists(LOAD_EXISTING_PROFILE_PATH):
    with open(LOAD_EXISTING_PROFILE_PATH, 'r', encoding='utf-8') as f:
        character_profile = json.load(f)
    print(f"✅ 既存プロファイルを読み込みました: {LOAD_EXISTING_PROFILE_PATH}")
    print(f"   character={character_profile['character_name']}, "
          f"ref_count={character_profile['reference_count']}, "
          f"core_tags={len(character_profile['identity_core_tags'])}")
else:
    ref_paths = list_image_files(REFERENCE_DIR)
    print(f"リファレンス画像数: {len(ref_paths)} ({REFERENCE_DIR})")
    character_profile, ref_records = build_character_profile(
        reference_paths=ref_paths,
        character_name=CHARACTER_NAME,
        core_presence_threshold=CORE_PRESENCE_THRESHOLD,
        core_mean_threshold=CORE_MEAN_THRESHOLD,
        tag_active_threshold=TAG_ACTIVE_THRESHOLD,
        kpt_thr=KPT_THR,
        force_identity_tags=FORCE_IDENTITY_TAGS,
        exclude_identity_tags=EXCLUDE_IDENTITY_TAGS,
    )
    with open(PROFILE_JSON_PATH, 'w', encoding='utf-8') as f:
        json.dump(character_profile, f, ensure_ascii=False, indent=2)
    print(f"✅ Character Profile を保存しました -> {PROFILE_JSON_PATH}")
    print(f"   抽出されたキャラ固有不変タグ数: {len(character_profile['identity_core_tags'])}")

show_profile_summary(character_profile, top_n=20)

## 6. 対象画像の変数化 & 一貫性チェック実行

立ち絵差分やアニメーション用フレーム（`TARGET_DIR`）を一括で変数化し、Section 5 で構築した `character_profile` に対する一貫性チェックを実行します。

- `image_variables.json`: 全対象画像の完全な変数定義（骨格＋WD14スロット＋一貫性評価）
- `image_variables_summary.csv`: 立ち絵・フレーム一覧管理用のテーブルデータ

In [ ]:
# ==========================================
# パラメータ設定（対象画像の変数化・評価）
# ==========================================
TARGET_DIR = './data/targets'
OUTPUT_JSON_PATH = './image_variables.json'
OUTPUT_CSV_PATH = './image_variables_summary.csv'
DEBUG_SKEL_DIR = './debug_skeletons'

# 一貫性チェックのアラート感度設定
IDENTITY_Z_WARN = 2.0            # 固有特徴タグが平均から何σ以上低下したら警告するか
PROPORTION_Z_WARN = 2.5          # 骨格プロポーションが平均から何σ以上ズレたら警告するか
UNEXPECTED_TAG_THRESHOLD = 0.55  # 学習分布にないタグがこの信頼度以上で出たら外乱警告するか

# ==========================================
# 一括解析の実行
# ==========================================
target_paths = list_image_files(TARGET_DIR)
assert len(target_paths) > 0, f"対象画像が見つかりません: {TARGET_DIR}"
os.makedirs(DEBUG_SKEL_DIR, exist_ok=True)

target_results = []
debug_skels = {}
summary_rows = []

for idx, path in enumerate(target_paths, 1):
    rec, wd14_pred, _, debug_skel = extract_image_variables(
        path,
        kpt_thr=KPT_THR,
        general_threshold=TAG_ACTIVE_THRESHOLD,
        return_debug_artifacts=True,
    )
    consistency = evaluate_against_profile(
        record=rec,
        wd14_pred=wd14_pred,
        profile=character_profile,
        identity_z_warn=IDENTITY_Z_WARN,
        proportion_z_warn=PROPORTION_Z_WARN,
        unexpected_tag_threshold=UNEXPECTED_TAG_THRESHOLD,
    )

    skel_path = os.path.join(DEBUG_SKEL_DIR, f"{rec['image_id']}_skeleton.png")
    cv2.imwrite(skel_path, debug_skel)
    debug_skels[rec['image_id']] = debug_skel

    combined = {**rec, 'consistency': consistency}
    target_results.append(combined)
    summary_rows.append(variables_to_flat_row(rec, consistency))
    print(f"[Target {idx}/{len(target_paths)}] {rec['file_name']} | "
          f"overall={consistency['overall_consistency']} "
          f"(id={consistency['identity_consistency']}, prop={consistency['proportion_consistency']})")

# JSON / CSV 保存
payload = {
    'schema_version': '1.0',
    'character_name': character_profile.get('character_name'),
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
    'target_count': len(target_results),
    'images': target_results,
}
with open(OUTPUT_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(payload, f, ensure_ascii=False, indent=2)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_CSV_PATH, index=False, encoding='utf-8-sig')
print(f"\n✅ 変数定義 JSON 保存: {OUTPUT_JSON_PATH}")
print(f"✅ 一覧サマリー CSV 保存: {OUTPUT_CSV_PATH}")

display(summary_df)

In [ ]:
# 各対象画像の詳細レポート（骨格・可変スロット・キャラ固有タグ照合）を表示
MAX_REPORTS_TO_SHOW = 10  # 画像が多い場合は先頭N件のみ表示

for item in target_results[:MAX_REPORTS_TO_SHOW]:
    print(f"\n==================== {item['file_name']} ====================")
    show_target_report(item, item['consistency'], debug_skels[item['image_id']])

# 複数画像（立ち絵差分・アニメ連番）の表情・視線スコア推移をヒートマップで比較
show_batch_variable_heatmap(target_results, slot_name='expression', min_max_score=0.20)
show_batch_variable_heatmap(target_results, slot_name='gaze', min_max_score=0.20)

## 7. 結果のDrive保存・ランタイム終了メモ

構築したキャラクター固有分布（`character_profile.json`）や対象画像の変数定義（`image_variables.json`, `image_variables_summary.csv`）を Google Drive へ保存します。

In [ ]:
# ==========================================
# Google Drive への結果保存
# ==========================================
DRIVE_OUTPUT_DIR = '/content/drive/MyDrive/Colab Notebooks/img/variables_output'
OUTPUT_PREFIX = f"{CHARACTER_NAME}_"

TARGET_FILES = [
    PROFILE_JSON_PATH,
    OUTPUT_JSON_PATH,
    OUTPUT_CSV_PATH,
]

save_results_to_drive(DRIVE_OUTPUT_DIR, file_paths=TARGET_FILES, prefix=OUTPUT_PREFIX)

### VS Code 上でのランタイム終了手順
作業が終わったら、GPUランタイムの課金・リソース消費を防ぐため、必ずランタイムを切断・削除してください。

1. VS Code 上部のコマンドパレット（`Ctrl + Shift + P` / macOS は `Cmd + Shift + P`）を開く
2. **`Colab: Remove Server`** を入力して実行する
3. 確認ダイアログが出たら対象の Colab ランタイムを削除して終了する